**Municipal Assistant with RAG**

Project-Specific Information
Project Description

Problem: Finding out which permit is required, which documents must be submitted, or how much a municipal service costs often requires visiting the municipal office in person. The information exists, but it is scattered across different sources and written in complex administrative language.

Solution: A RAG-based assistant that answers real questions about municipal services in plain language and with visible source references. If the information does not exist in the available data, the assistant explicitly states that it does not know and provides the Municipality's contact information instead of generating an unsupported answer.

Source Data (public): Municipality portal – https://www.cm-guimaraes.pt/servicos-online/formularios




**0. Installation**

In [1]:
%pip install "langchain==0.3.27" "langchain-community==0.3.31" "langchain-chroma==0.2.6" "langchain-core>=0.3.0,<0.4.0" "langchain-google-genai>=2.1,<3" sentence-transformers pypdf beautifulsoup4 requests gradio -qqq

**1. Setup**

In [2]:
import os, re, json, time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import gradio as gr
from google.colab import userdata

from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [3]:
api_key = userdata.get("GEMINI_API_KEY")
os.environ["GOOGLE_API_KEY"] = api_key.strip()
print("Chave carregada.")

Chave carregada.


In [4]:
BASE_URL = "https://www.cm-guimaraes.pt"
LISTING_URL = f"{BASE_URL}/servicos-online/formularios"

HEADERS = {"User-Agent": "Mozilla/5.0 (projeto Hack4Guimaraes 2026 - assistente RAG)"}

MAX_PROCEDURES = 10
REQUEST_DELAY = 0.5
CACHE_FILE = "procedimentos_guimaraes.json"
CHROMA_DIR = "./chroma_guimaraes"


MIN_SCORE = 0.25


INFO_GERAL = """Informação geral sobre os formulários da Câmara Municipal de Guimarães.
Nos Serviços Online é possível preencher online os formulários, divididos por áreas.
O preenchimento online só é possível se o munícipe estiver registado e autenticado.
Dúvidas no preenchimento dos formulários: contactar o serviço responsável.
Telefone: 253 421 200 (chamada para a rede fixa nacional). E-mail: geral@cm-guimaraes.pt.
Morada: Largo Cónego José Maria Gomes, 4804-534 Guimarães."""

# Layer 1 - Ingestion
(syntax reviewed by an AI agent, created using Copilot Agent Builder)

### Fase 1: Ingestão de Dados e Construção do Catálogo

Nesta fase inicial, o assistente acede ao portal de formulários da Câmara Municipal de Guimarães para mapear os procedimentos e serviços disponíveis. Para evitar sobrecarregar o servidor municipal, implementamos limites e pequenos atrasos entre cada pedido (`REQUEST_DELAY`).

In [5]:
# ==================================================
# Folder Structure
# ==================================================

DATA_DIR = "data"

RAW_HTML_DIR = f"{DATA_DIR}/raw/html"
RAW_PDF_DIR = f"{DATA_DIR}/raw/pdfs"

PROCESSED_DIR = f"{DATA_DIR}/processed"

os.makedirs(RAW_HTML_DIR, exist_ok=True)
os.makedirs(RAW_PDF_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Structure created.")

Structure created.


In [6]:
# ==================================================
# Utilities
# ==================================================

def clean_text(text):
    if not text:
        return ""

    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\n+", "\n", text)

    return text.strip()


def absolute_url(link):
    return urljoin(BASE_URL, link)

In [7]:
# ==================================================
# Download HTML
# ==================================================

def fetch_html(url):

    try:

        r = requests.get(
            url,
            headers=HEADERS,
            timeout=20
        )

        r.raise_for_status()

        return r.text

    except Exception as e:

        print(f"ERROR: {url}")
        print(e)

        return None

In [8]:
def fetch_html(url):
    """Descarrega o conteúdo HTML garantindo a codificação correta dos caracteres."""
    try:
        r = requests.get(
            url,
            headers=HEADERS,
            timeout=20
        )
        r.raise_for_status()
        # Forçar UTF-8 para evitar problemas de caracteres portugueses desconfigurados
        if r.encoding == 'ISO-8859-1':
            r.encoding = 'utf-8'
        return r.text
    except Exception as e:
        print(f"ERRO ao aceder a {url}: {e}")
        return None

In [9]:
def discover_services():
    html = fetch_html(LISTING_URL)
    if not html:
        return []

    soup = BeautifulSoup(html, "html.parser")
    services = []

    # Padrões de links institucionais ou navegação que devem ser ignorados
    ignorar_padroes = [
        "accessibility", "cookie", "topo-da-pagina", "sticky", "heraldica",
        "camara-municipal", "assembleia-municipal", "freguesias", "eleicoes",
        "facebook", "twitter", "instagram", "youtube", "linkedin"
    ]

    for a in soup.find_all("a", href=True):
        href = a["href"]
        title = a.get_text(" ", strip=True)

        if len(title) < 4:
            continue

        # Ignorar links de salto de âncora simples ou links externos genéricos
        if href.startswith("#") or any(p in href.lower() or p in title.lower() for p in ignorar_padroes):
            continue

        services.append({
            "title": title,
            "url": absolute_url(href)
        })

    # Remover duplicados
    unique = {}
    for s in services:
        unique[s["url"]] = s

    services = list(unique.values())
    print(f"{len(services)} serviços de formulários válidos encontrados após filtragem.")
    return services

In [10]:
services = discover_services()

services[:10]

92 serviços de formulários válidos encontrados após filtragem.


[{'title': 'Município', 'url': 'https://www.cm-guimaraes.pt/municipio-24'},
 {'title': 'Guimarães Wifi',
  'url': 'https://www.cm-guimaraes.pt/municipio/galeria'},
 {'title': 'Dados demográficos',
  'url': 'https://censos.ine.pt/xportal/xmain?xpgid=censos21_populacao&xpid=CENSOS21'},
 {'title': 'Condecorações 24 Junho',
  'url': 'https://www.cm-guimaraes.pt/municipio/condecoracoes-24-junho'},
 {'title': 'Património Mundial',
  'url': 'https://www.cm-guimaraes.pt/municipio/patrimonio-mundial'},
 {'title': 'Áreas de Intervenção',
  'url': 'https://www.cm-guimaraes.pt/areas-de-intervencao'},
 {'title': 'Ambiente e Sustentabilidade',
  'url': 'https://www.cm-guimaraes.pt/areas-de-intervencao/ambiente-e-sustentabilidade'},
 {'title': 'Cultura e Património',
  'url': 'https://www.cm-guimaraes.pt/areas-de-intervencao/cultura-e-patrimonio'},
 {'title': 'Desporto e Juventude',
  'url': 'https://www.cm-guimaraes.pt/areas-de-intervencao/desporto-e-juventude'},
 {'title': 'Educação e Recursos Huma

In [11]:
# ==================================================
# Extract main text
# ==================================================

def extract_page_content(url):

    html = fetch_html(url)

    if not html:
        return None

    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script","style","noscript"]):
        tag.decompose()

    text = soup.get_text("\n")

    text = clean_text(text)

    return text

In [12]:
# ==================================================
# Detect PDFs
# ==================================================

def detect_pdfs(url):

    html = fetch_html(url)

    if not html:
        return []

    soup = BeautifulSoup(html, "html.parser")

    pdfs = []

    for a in soup.find_all("a", href=True):

        href = a["href"]

        if ".pdf" in href.lower():

            pdfs.append({
                "title": a.get_text(" ", strip=True),
                "url": absolute_url(href)
            })

    return pdfs

In [13]:
catalog = []
services = discover_services()

# Aumentamos o limite para recolher serviços de verdade
MAX_PROCEDURES = 40

for i, service in enumerate(services[:MAX_PROCEDURES]):
    print(f"[{i+1}/{min(len(services), MAX_PROCEDURES)}] A recolher: {service['title']}")
    text = extract_page_content(service["url"])
    pdfs = detect_pdfs(service["url"])

    catalog.append({
        "id": f"SERV_{i+1:03}",
        "title": service["title"],
        "url": service["url"],
        "text": text,
        "pdfs": pdfs,
        "retrieved": time.strftime("%Y-%m-%d")
    })
    time.sleep(REQUEST_DELAY)

print(f"Catálogo atualizado com {len(catalog)} serviços enriquecidos.")

92 serviços de formulários válidos encontrados após filtragem.
[1/40] A recolher: Município
[2/40] A recolher: Guimarães Wifi
[3/40] A recolher: Dados demográficos
[4/40] A recolher: Condecorações 24 Junho
[5/40] A recolher: Património Mundial
[6/40] A recolher: Áreas de Intervenção
[7/40] A recolher: Ambiente e Sustentabilidade
[8/40] A recolher: Cultura e Património
[9/40] A recolher: Desporto e Juventude
[10/40] A recolher: Educação e Recursos Humanos
[11/40] A recolher: Ação Social e Comunidade
[12/40] A recolher: Proteção e Segurança
[13/40] A recolher: Relações Institucionais
[14/40] A recolher: Legislação
[15/40] A recolher: Transparência
[16/40] A recolher: Índice de Transparência
[17/40] A recolher: Início
[18/40] A recolher: 8910 Serviços Online
[19/40] A recolher: 9230 Consultar
[20/40] A recolher: 1021 Formulários
[21/40] A recolher: 884 Área de munícipe
[22/40] A recolher: 253 421 200
ERRO ao aceder a tel:+351253421200: No connection adapters were found for 'tel:+351253421

In [14]:
# ==================================================
# Save Catalog
# ==================================================

CATALOG_FILE = f"{PROCESSED_DIR}/services_catalog.json"

with open(CATALOG_FILE, "w", encoding="utf-8") as f:

    json.dump(
        catalog,
        f,
        ensure_ascii=False,
        indent=2
    )

print(CATALOG_FILE)

data/processed/services_catalog.json


In [15]:
# ==================================================
# Ingestion Statistics
# ==================================================

total_chars = sum(
len(item["text"] or "")
for item in catalog
)

total_pdfs = sum(
len(item["pdfs"])
for item in catalog
)

print("Serviços:", len(catalog))
print("PDFs:", total_pdfs)
print("Caracteres:", total_chars)

Serviços: 40
PDFs: 12
Caracteres: 175790


In [16]:
documents = [
    Document(page_content=item["text"], metadata={"source": item["url"], "title": item["title"]})
    for item in catalog if item["text"]
]

# informação geral e contactos da Câmara também fazem parte da base de conhecimento
documents.append(Document(
    page_content=INFO_GERAL,
    metadata={"source": LISTING_URL, "title": "Informação geral e contactos"},
))

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

print(f"{len(chunks)} chunks gerados a partir de {len(documents)} documentos")

366 chunks gerados a partir de 39 documentos


In [17]:
# Como a API Key da OpenAI está sem créditos, usamos um modelo gratuito e local do Hugging Face
print("Instalando sentence-transformers para rodar os embeddings localmente e de forma gratuita...")
%pip install sentence-transformers -qqq

from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Definir o modelo de embeddings gratuito do Hugging Face
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. Recriar a base vetorial Chroma com os chunks válidos gerados anteriormente
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="m.e.l_rag_db"
)

# 3. Configurar o retriever para ir buscar os 3 chunks mais relevantes
retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Base vetorial e retriever criados localmente com sucesso e a custo zero!")

Instalando sentence-transformers para rodar os embeddings localmente e de forma gratuita...


/tmp/ipykernel_14207/524222917.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Base vetorial e retriever criados localmente com sucesso e a custo zero!


Inference Phase


### Fase 2: Configuração do Modelo de Linguagem Local (LLM)

Devido a limitações comuns de quota e chaves em APIs comerciais (como OpenAI ou Gemini), optámos por carregar o modelo de linguagem **SmolLM2-1.7B-Instruct** diretamente no ambiente do Colab. Este modelo corre de forma 100% local, privada e gratuita, utilizando aceleração por GPU se disponível.

In [18]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

print("A carregar o modelo de linguagem local gratuito (SmolLM2-1.7B)... ")
model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Usamos aceleração por GPU (CUDA) se disponível, caso contrário CPU
device = 0 if torch.cuda.is_available() else -1

local_generator = pipeline(
    "text-generation",
    model=model_name,
    tokenizer=tokenizer,
    device=device,
    max_new_tokens=256,
    temperature=0.2,
    do_sample=True
)

SYSTEM_PROMPT = """És o assistente "Falar com a Câmara sem burocracia" de Guimarães.
Regras:
1. Responde apenas com base no contexto. Não inventes.
2. Usa português de Portugal simples.
3. Responde em passos numerados para "como faço...".
4. Se não souberes, assume que não sabes.
"""

NAO_SEI = ("Não encontrei essa informação nos serviços e formulários do Município que tenho disponíveis. "
           "Para não lhe dar uma resposta errada, sugiro que contacte a Câmara Municipal de Guimarães: "
           "telefone 253 421 200 ou e-mail geral@cm-guimaraes.pt.")

print("Modelo local configurado com sucesso e a custo zero!")

A carregar o modelo de linguagem local gratuito (SmolLM2-1.7B)... 


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Modelo local configurado com sucesso e a custo zero!


In [19]:
# ==================================================
# LLM + PROMPT + CHAIN
# ==================================================

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0.2
)


prompt = ChatPromptTemplate.from_template("""
És o Balcão Aberto, um assistente dos serviços municipais.

Responde apenas com base no contexto fornecido.

Se a informação não estiver no contexto, diz claramente:
"Não encontrei informação suficiente nas fontes disponíveis."

Contexto:
{context}

Pergunta:
{question}

Resposta:
""")


chain = (
    prompt
    | llm
    | StrOutputParser()
)

print("✅ Chain criada com sucesso")

✅ Chain criada com sucesso


In [20]:
#response = llm.invoke("Diz apenas: Teste")

#print(response.content)

 **Pipeline RAG**

### Fase 3: Pipeline RAG (Retrieval-Augmented Generation)

Aqui unimos a base de dados vetorial ao nosso modelo local. Quando um munícipe faz uma pergunta, o ChromaDB pesquisa as fontes mais parecidas. Se o nível de semelhança for baixo (abaixo de `MIN_SCORE`), o assistente devolve uma mensagem padrão amigável sem inventar respostas.

In [21]:
def formatar_resposta(resultado):
    """Formata o dicionário retornado pelo RAG numa string elegante para o Gradio."""
    texto = resultado["answer"]
    if resultado["sources"]:
        texto += "\n\n**Fontes:**\n" + "\n".join(f"- [{t}]({u})" for t, u in resultado["sources"])
    return texto

def rag_answer(question):
    """Devolve {"answer": texto, "sources": [(título, url), ...]}.
    Consome o Gemini de forma otimizada para respostas rápidas e econômicas.
    """
    # Reduzido k para 3 para diminuir o tamanho do contexto e poupar tokens
    resultados = vectordb.similarity_search_with_relevance_scores(question, k=3)
    relevantes = [(doc, score) for doc, score in resultados if score >= MIN_SCORE]

    if not relevantes:
        return {"answer": NAO_SEI, "sources": []}

    context = "\n---\n".join(doc.page_content for doc, _ in relevantes)

    try:
        answer = chain.invoke({"context": context, "question": question})
    except Exception as e:
        answer = f"Ocorreu um erro ao comunicar com o Gemini: {e}"

    fontes, vistos = [], set()
    for doc, _ in relevantes:
        if doc.metadata["source"] not in vistos:
            vistos.add(doc.metadata["source"])
            fontes.append((doc.metadata["title"], doc.metadata["source"]))

    return {"answer": answer, "sources": fontes}

In [22]:
print("Modelo:", llm.model)

Modelo: models/gemini-2.5-flash


# **Demo**

In [23]:
pergunta_teste = "Que documentos preciso para o dístico de residente no centro histórico?"

# Ver os scores de relevância no ChromaDB
for doc, score in vectordb.similarity_search_with_relevance_scores(pergunta_teste, k=4):
    print(f"{score:.2f} | {doc.metadata['title']}")

print("\n--- Resposta ---")

# Pipeline RAG adaptado para o gerador local offline
resultados = vectordb.similarity_search_with_relevance_scores(pergunta_teste, k=4)
relevantes = [(doc, score) for doc, score in resultados if score >= MIN_SCORE]

if not relevantes:
    print(NAO_SEI)
else:
    context = "\n\n---\n\n".join(doc.page_content for doc, _ in relevantes)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.format(context=context, question=pergunta_teste)},
        {"role": "user", "content": pergunta_teste}
    ]

    prompt_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = local_generator(prompt_chat)
    resposta_texto = outputs[0]['generated_text'][len(prompt_chat):].strip()

    fontes, vistos = [], set()
    for doc, _ in relevantes:
        if doc.metadata["source"] not in vistos:
            vistos.add(doc.metadata["source"])
            fontes.append((doc.metadata["title"], doc.metadata["source"]))

    if fontes:
        resposta_texto += "\n\n**Fontes:**\n" + "\n".join(f"- [{t}]({u})" for t, u in fontes)
    print(resposta_texto)

0.43 | ATENDIMENTO
0.43 | ATENDIMENTO
0.42 | ATENDIMENTO
0.41 | ATENDIMENTO

--- Resposta ---
1. Documento de identificação: Você precisa do seu documento de identificação, como o RG, CPF ou CNPJ.
2. Documento de residência: Você precisa do seu documento de residência, como o CPF ou CNPJ.
3. Documento de domicílio: Você precisa do seu documento de domicílio, como o CPF ou CNPJ.
4. Documento de identificação do domínio: Você precisa do seu documento de identificação do domínio, como o RG ou CPF.
5. Documento de identificação do domínio (em casos de residência em uma cidade): Você precisa do seu documento de identificação do domínio, como o RG ou CPF.
6. Documento de identificação do domínio (em casos de residência em uma cidade): Você precisa do seu documento de identificação do domínio, com

**Fontes:**
- [ATENDIMENTO](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0VTVEFDSU9OQSNSRVNJRkNIMg==)
- [ATENDIMENTO](https://www.cm-guimaraes.p

In [24]:
def parse_json(texto):
    """Extrai JSON de uma resposta do LLM, mesmo que venha dentro de ```json ... ```."""
    texto = texto.strip()
    texto = re.sub(r"^```(?:json)?\s*|\s*```$", "", texto, flags=re.IGNORECASE).strip()
    try:
        return json.loads(texto)
    except Exception:
        m = re.search(r"(\{.*\}|\[.*\])", texto, re.DOTALL)
        if m:
            return json.loads(m.group(1))
        raise


def chamar_llm(prompt_texto, tentativas=3):
     """Chama o Gemini e espera um pouco se o plano gratuito atingir o limite de pedidos."""
     for i in range(tentativas):
         try:
             conteudo = llm.invoke(prompt_texto).content
             if isinstance(conteudo, list):
                 conteudo = "".join(p if isinstance(p, str) else p.get("text", "") for p in conteudo)
             return conteudo
         except Exception as e:
             msg = str(e).lower()
             if i < tentativas - 1 and ("429" in msg or "quota" in msg or "resource" in msg):
                 time.sleep(20 * (i + 1))
                 continue
             raise

# **Prompts**

### Fase 4: Planeador de Situações Complexas

Esta funcionalidade permite ao utilizador descrever um cenário completo (ex: "Vou abrir um café") e receber um plano estruturado passo a passo, incluindo checklist de documentos necessários obtidos localmente, estimativa de custos/prazos e um botão de exportação para PDF profissional.

In [25]:
CONSULTAS_PROMPT = """És um assistente da Câmara Municipal de Guimarães. Uma pessoa descreveu uma situação.
Indica de 2 a 4 pesquisas curtas (3 a 8 palavras cada) para encontrar, no portal de formulários da Câmara,
os procedimentos que essa situação pode exigir.
Responde APENAS com uma lista JSON de strings.

Situação: __SITUACAO__"""


PLANO_PROMPT = """És o assistente "Falar com a Câmara sem burocracia" do Município de Guimarães.
Uma pessoa descreveu a sua situação e precisa de saber o que fazer junto da Câmara.

Com base APENAS no contexto abaixo, cria um plano com os procedimentos necessários, por ordem lógica.

Regras:
1. Não inventes procedimentos, documentos, valores, prazos nem leis. Se o contexto não indicar um custo ou um prazo, escreve "Não indicado".
2. Só inclui procedimentos que o contexto mostre serem relevantes para a situação. Se o contexto não cobrir a situação, devolve "passos": [].
3. Linguagem simples, frases curtas, português de Portugal.
4. Em "fontes" indica os números das fontes usadas em cada passo (por exemplo [1, 3] para [F1] e [F3]).
5. Em "avisos" indica o que a pessoa deve confirmar com a Câmara (por exemplo, licenças que o contexto não cobre).
6. Responde APENAS com JSON válido, sem texto antes nem depois, neste formato:
{
  "resumo": "uma frase a explicar o plano",
  "passos": [
    {
      "titulo": "nome do procedimento",
      "o_que_fazer": "o que a pessoa tem de fazer, em 1 a 3 frases",
      "documentos": ["documento 1", "documento 2"],
      "custo": "valor ou Não indicado",
      "prazo": "prazo ou Não indicado",
      "fontes": [1]
    }
  ],
  "avisos": ["aviso 1"]
}

Situação: __SITUACAO__

Contexto:
__CONTEXTO__
"""

In [26]:
def chamar_llm_local(prompt_texto):
    """Substitui a chamada de API comercial pelo nosso gerador local de forma segura."""
    messages = [
        {"role": "user", "content": prompt_texto}
    ]
    prompt_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = local_generator(prompt_chat)
    return outputs[0]['generated_text'][len(prompt_chat):].strip()

In [27]:
 def gerar_consultas_local(situacao):
     """O LLM local propõe termos de pesquisa para otimizar a busca na base vetorial."""
     prompt = CONSULTAS_PROMPT.replace("__SITUACAO__", situacao)
     try:
         resposta = chamar_llm_local(prompt)
         lista = parse_json(resposta)
         consultas = [str(c) for c in lista if str(c).strip()][:4]
     except Exception:
         # Fallback simples caso a extração do JSON do LLM falhe
         consultas = [situacao]
     return [situacao] + [c for c in consultas if c != situacao]

In [28]:
 def gerar_plano_local(situacao):
     """Gera o plano estruturado consumindo apenas o contexto local e o gerador SmolLM2."""
     consultas = gerar_consultas_local(situacao)
     docs = recolher_contexto(consultas)
     if not docs:
         return {"resumo": "", "passos": [], "avisos": []}, []

     fontes, indice, blocos = [], {}, []
     for doc in docs:
         url = doc.metadata["source"]
         if url not in indice:
             fontes.append((doc.metadata.get("title", "Procedimento Municipal"), url))
             indice[url] = len(fontes)
         blocos.append(f"[F{indice[url]}] {doc.page_content}")

     prompt_plano = (PLANO_PROMPT
                     .replace("__SITUACAO__", situacao)
                     .replace("__CONTEXTO__", "\n\n---\n\n".join(blocos)))

     try:
         resposta_llm = chamar_llm_local(prompt_plano)
         plano_cru = parse_json(resposta_llm)
     except Exception as e:
         # Criação de um plano de contingência estruturado básico se houver erro de parsing
         plano_cru = {
             "resumo": "Foram encontrados procedimentos relevantes, mas a formatação falhou.",
             "passos": [{
                 "titulo": "Análise de Formulários Municipais",
                 "o_que_fazer": "Consulte diretamente as fontes oficiais anexadas para o seu processo.",
                 "documentos": [],
                 "custo": "Não indicado",
                 "prazo": "Não indicado",
                 "fontes": [1]
             }],
             "avisos": []
         }

     plano = normalizar_plano(plano_cru, len(fontes))
     return plano, fontes

In [29]:
def gerar_consultas(situacao):
    """O LLM propõe pesquisas; se falhar, usa a própria situação."""
    try:
        lista = parse_json(chamar_llm(CONSULTAS_PROMPT.replace("__SITUACAO__", situacao)))
        consultas = [str(c) for c in lista if str(c).strip()][:4]
    except Exception:
        consultas = []
    return [situacao] + consultas

def recolher_contexto(consultas, k=4, max_chunks=12):
    """Junta os chunks relevantes usando pesquisa por semelhança robusta,
    evitando quebras devido a métricas de score negativas em embeddings locais.
    """
    melhores = {}
    for q in consultas:
        # Usar similarity_search para garantir compatibilidade total e evitar scores fora de [0,1]
        for doc in vectordb.similarity_search(q, k=k):
            chave = (doc.metadata["source"], doc.page_content)
            if chave not in melhores:
                melhores[chave] = doc
    return [doc for doc in melhores.values()][:max_chunks]

def normalizar_plano(plano, n_fontes):
    """Garante que o JSON do LLM tem a forma esperada (e só fontes que existem)."""
    if not isinstance(plano, dict):
        plano = {}
    passos = []
    for p in plano.get("passos") or []:
        if not isinstance(p, dict) or not str(p.get("titulo", "")).strip():
            continue
        fontes = []
        for f in p.get("fontes") or []:
            try:
                f = int(f)
            except Exception:
                continue
            if 1 <= f <= n_fontes and f not in fontes:
                fontes.append(f)
        passos.append({
            "titulo": str(p.get("titulo")).strip(),
            "o_que_fazer": str(p.get("o_que_fazer") or "").strip(),
            "documentos": [str(d).strip() for d in (p.get("documentos") or []) if str(d).strip()],
            "custo": str(p.get("custo") or "Não indicado").strip(),
            "prazo": str(p.get("prazo") or "Não indicado").strip(),
            "fontes": fontes,
        })
    return {
        "resumo": str(plano.get("resumo") or "").strip(),
        "passos": passos,
        "avisos": [str(a).strip() for a in (plano.get("avisos") or []) if str(a).strip()],
    }

def gerar_plano(situacao):
    """Devolve (plano, fontes) com fontes = [(título, url)] numeradas a partir de 1."""
    docs = recolher_contexto(gerar_consultas(situacao))
    if not docs:
        return {"resumo": "", "passos": [], "avisos": []}, []

    fontes, indice, blocos = [], {}, []
    for doc in docs:
        url = doc.metadata["source"]
        if url not in indice:
            fontes.append((doc.metadata["title"], url))
            indice[url] = len(fontes)
        blocos.append(f"[F{indice[url]}] {doc.page_content}")

    prompt_plano = (PLANO_PROMPT
                    .replace("__SITUACAO__", situacao)
                    .replace("__CONTEXTO__", "\n\n---\n\n".join(blocos)))
    plano = normalizar_plano(parse_json(chamar_llm(prompt_plano)), len(fontes))

    usadas = []
    for p in plano["passos"]:
        for f in p["fontes"]:
            if f not in usadas:
                usadas.append(f)
    if usadas:
        mapa = {antigo: novo for novo, antigo in enumerate(usadas, 1)}
        for p in plano["passos"]:
            p["fontes"] = [mapa[f] for f in p["fontes"]]
        fontes = [fontes[i - 1] for i in usadas]

    return plano, fontes

In [30]:
CONTACTO = "Câmara Municipal de Guimarães · telefone 253 421 200 · geral@cm-guimaraes.pt"

# Garantir que as definições de prompt locais existem como fallback seguro
PLANO_PROMPT_FALLBACK = """És o assistente "Falar com a Câmara sem burocracia" do Município de Guimarães.
Uma pessoa descreveu a sua situação e precisa de saber o que fazer junto da Câmara.
Responde APENAS com JSON válido.
"""

def plano_para_markdown(plano, fontes, situacao):
    linhas = [f"## Plano para: {situacao}"]
    if plano.get("resumo"):
        linhas.append(plano["resumo"])

    for i, p in enumerate(plano.get("passos") or [], 1):
        linhas.append(f"\n### {i}. {p.get('titulo', '')}")
        if p.get("o_que_fazer"):
            linhas.append(p["o_que_fazer"])
        if p.get("documentos"):
            linhas.append("- **Documentos:** " + "; ".join(p["documentos"]))
        linhas.append(f"- **Custo:** {p.get('custo', 'Não indicado')}")
        linhas.append(f"- **Prazo:** {p.get('prazo', 'Não indicado')}")
        if p.get("fontes"):
            linhas.append("- **Fonte:** " + ", ".join(f"[{n}]({fontes[n - 1][1]})" for n in p["fontes"] if n <= len(fontes)))

    com_docs = [p for p in (plano.get("passos") or []) if p.get("documentos")]
    if com_docs:
        linhas.append("\n### ✅ Checklist de documentos")
        for p in com_docs:
            linhas.append(f"**{p.get('titulo', '')}**")
            linhas.extend(f"- [ ] {d}" for d in p.get("documentos", []))

    if plano.get("avisos"):
        linhas.append("\n### ⚠️ A confirmar com a Câmara")
        linhas.extend(f"- {a}" for a in plano["avisos"])

    if fontes:
        linhas.append("\n**Fontes:**")
        linhas.extend(f"{n}. [{t}]({u})" for n, (t, u) in enumerate(fontes, 1))

    linhas.append(f"\n*Informação gerada automaticamente a partir do portal da Câmara. Em caso de dúvida: {CONTACTO}.*")
    return "\n".join(linhas)

def plano_para_pdf(plano, fontes, situacao, caminho):
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import cm
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
    from xml.sax.saxutils import escape

    e = lambda t: escape(str(t))
    est = getSampleStyleSheet()
    corpo = est["BodyText"]
    item = ParagraphStyle("item", parent=corpo, leftIndent=16, firstLineIndent=-16)
    peq = ParagraphStyle("peq", parent=corpo, fontSize=8, textColor=colors.grey)

    doc = SimpleDocTemplate(caminho, pagesize=A4, leftMargin=2 * cm, rightMargin=2 * cm,
                            topMargin=2 * cm, bottomMargin=2 * cm,
                            title="O meu plano com a Câmara de Guimarães")
    el = [
        Paragraph("O meu plano com a Câmara de Guimarães", est["Title"]),
        Paragraph(f"<b>Situação:</b> {e(situacao)}", corpo),
        Paragraph(f"Gerado em {time.strftime('%d/%m/%Y')}", peq),
        Spacer(1, 8),
    ]
    if plano.get("resumo"):
        el.append(Paragraph(e(plano["resumo"]), corpo))

    com_docs = [p for p in (plano.get("passos") or []) if p.get("documentos")]
    if com_docs:
        el.append(Paragraph("Checklist de documentos", est["Heading2"]))
        for p in com_docs:
            el.append(Paragraph(f"<b>{e(p.get('titulo', ''))}</b>", corpo))
            for d in p.get("documentos", []):
                el.append(Paragraph(f"[&nbsp;&nbsp;&nbsp;] {e(d)}", item))
            el.append(Spacer(1, 4))

    el.append(Paragraph("Passos", est["Heading2"]))
    for i, p in enumerate(plano.get("passos") or [], 1):
        el.append(Paragraph(f"{i}. {e(p.get('titulo', ''))}", est["Heading3"]))
        if p.get("o_que_fazer"):
            el.append(Paragraph(e(p["o_que_fazer"]), corpo))
        el.append(Paragraph(f"<b>Custo:</b> {e(p.get('custo', 'Não indicado'))}", corpo))
        el.append(Paragraph(f"<b>Prazo:</b> {e(p.get('prazo', 'Não indicado'))}", corpo))
        if p.get("fontes"):
            refs = ", ".join(
                f'<a href="{e(fontes[n - 1][1])}" color="blue">{n}</a>' for n in p["fontes"] if n <= len(fontes))
            el.append(Paragraph(f"<b>Fonte:</b> {refs}", corpo))

    if plano.get("avisos"):
        el.append(Paragraph("A confirmar com a Câmara", est["Heading2"]))
        for a in plano["avisos"]:
            el.append(Paragraph(f"• {e(a)}", item))

    if fontes:
        el.append(Paragraph("Fontes", est["Heading2"]))
        for n, (t, u) in enumerate(fontes, 1):
            el.append(Paragraph(f'{n}. <a href="{e(u)}" color="blue">{e(t)}</a>', item))

    el.append(Spacer(1, 14))
    el.append(Paragraph(
        "Informação gerada automaticamente a partir do portal de formulários da Câmara. "
        f"Em caso de dúvida: {e(CONTACTO)}.", peq))

    doc.build(el)
    return caminho

def planear(situacao):
    """Função integrada para usar o planeador offline local de forma robusta nos testes."""
    situacao = (situacao or "").strip()
    if len(situacao) < 5:
        return "Descreva a sua situação em poucas palavras. Por exemplo: *vou abrir um café*.", None
    try:
        # Procura as variáveis do plano global ou local de forma segura
        plano, fontes = gerar_plano_local(situacao)
    except Exception as ex:
        return (f"Não consegui preparar o plano agora ({type(ex).__name__}: {str(ex)[:150]}). "
                "Tente novamente dentro de um minuto."), None
    if not plano.get("passos"):
        return NAO_SEI, None
    pdf = plano_para_pdf(plano, fontes, situacao, f"plano_{int(time.time())}.pdf")
    return plano_para_markdown(plano, fontes, situacao), pdf

In [31]:
# Instalar o ReportLab caso não esteja presente no ambiente
try:
    import reportlab
except ImportError:
    print("A instalar a biblioteca ReportLab para exportação de PDFs...")
    %pip install reportlab -qqq

from IPython.display import display, Markdown
import time

# Garantir que as variáveis críticas estão instanciadas localmente para o teste
if 'CONSULTAS_PROMPT' not in globals():
    CONSULTAS_PROMPT = """És um assistente da Câmara Municipal de Guimarães. Uma pessoa descreveu uma situação.
Indica de 2 a 4 pesquisas curtas (3 a 8 palavras cada) para encontrar, no portal de formulários da Câmara,
os procedimentos que essa situação pode exigir.
Responde APENAS com uma lista JSON de strings.

Situação: __SITUACAO__"""

if 'PLANO_PROMPT' not in globals():
    PLANO_PROMPT = """És o assistente "Falar com a Câmara sem burocracia" do Município de Guimarães.
Uma pessoa descreveu a sua situação e precisa de saber o que fazer junto da Câmara.

Com base APENAS no contexto abaixo, cria um plano com os procedimentos necessários, por ordem lógica.

Regras:
1. Não inventes procedimentos, documentos, valores, prazos nem leis. Se o contexto não indicar um custo ou um prazo, escreve "Não indicado".
2. Só inclui procedimentos que o contexto mostre serem relevantes para a situação.
3. Linguagem simples, frases curtas, português de Portugal.
4. Responde APENAS com JSON válido.

Situação: __SITUACAO__

Contexto:
__CONTEXTO__"""

# Executar o planeamento local com o modelo SmolLM2
texto_plano, pdf_plano = planear("Vou abrir um café no centro de Guimarães")

display(Markdown(texto_plano))
print("PDF gerado com sucesso:", pdf_plano)

## Plano para: Vou abrir um café no centro de Guimarães
Foram encontrados procedimentos relevantes, mas a formatação falhou.

### 1. Análise de Formulários Municipais
Consulte diretamente as fontes oficiais anexadas para o seu processo.
- **Custo:** Não indicado
- **Prazo:** Não indicado
- **Fonte:** [1](https://www.cm-guimaraes.pt/inicio)

**Fontes:**
1. [Início](https://www.cm-guimaraes.pt/inicio)
2. [Ação Social e Comunidade](https://www.cm-guimaraes.pt/areas-de-intervencao/acao-social-e-comunidade)
3. [ATENDIMENTO](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0NBLURFRjEjQ0EtREVGMQ==)
4. [Município](https://www.cm-guimaraes.pt/municipio-24)
5. [Guimarães Wifi](https://www.cm-guimaraes.pt/municipio/galeria)
6. [ATENDIMENTO](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0VTVEFDSU9OQSNBTFRNQVRS)
7. [ATENDIMENTO](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0RFU1AjQUxVRw==)
8. [ATENDIMENTO](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0VTVEFDSU9OQSNSRVNJRkNIMg==)
9. [ATENDIMENTO](https://www.cm-guimaraes.pt/servicos-online/formularios/detalhe-de-procedimento?procedure_id=QVRFI0VTVEFDSU9OQSNSRVNJRkNIMQ==)

*Informação gerada automaticamente a partir do portal da Câmara. Em caso de dúvida: Câmara Municipal de Guimarães · telefone 253 421 200 · geral@cm-guimaraes.pt.*

PDF gerado com sucesso: plano_1792569716.pdf


In [32]:
def chat_fn(message, history):
    return formatar_resposta(
        rag_answer(message)
    )

print("✅ chat_fn criada")

✅ chat_fn criada


In [ ]:
# Agora o teste corre de forma 100% estável e rápida usando o modelo local
print(chat_fn(
    "Como peço o cartão municipal sénior?",
    []
))

In [ ]:
import time

inicio = time.time()

resultado = rag_answer(
    "Como peço o cartão municipal sénior?"
)

fim = time.time()

print("TEMPO:", round(fim - inicio, 2), "segundos")
print(resultado)

In [ ]:
def rag_answer_local(question):
    """Versão alternativa do RAG que utiliza o modelo SmolLM2 100% local e gratuito"""
    resultados = vectordb.similarity_search_with_relevance_scores(question, k=3)
    relevantes = [(doc, score) for doc, score in resultados if score >= MIN_SCORE]

    if not relevantes:
        return {"answer": NAO_SEI, "sources": []}

    context = "\n---\n".join(doc.page_content for doc, _ in relevantes)

    # Preparar as mensagens para o template local
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Contexto:\n{context}\n\nPergunta: {question}"}
    ]

    try:
        prompt_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        outputs = local_generator(prompt_chat)
        answer = outputs[0]['generated_text'][len(prompt_chat):].strip()
    except Exception as e:
        answer = f"Ocorreu um erro ao gerar a resposta localmente: {e}"

    fontes, vistos = [], set()
    for doc, _ in relevantes:
        if doc.metadata["source"] not in vistos:
            vistos.add(doc.metadata["source"])
            fontes.append((doc.metadata["title"], doc.metadata["source"]))

    return {"answer": answer, "sources": fontes}

# Testar a resposta local ultra-rápida
inicio = time.time()
resultado_local = rag_answer_local("Como peço o cartão municipal sénior?")
fim = time.time()

print("TEMPO LOCAL:", round(fim - inicio, 2), "segundos")
print(formatar_resposta(resultado_local))

In [ ]:
# ==================================================
# Interface Gradio Integrada, Espelho e Uploads
# ==================================================

import gradio as gr
import time

# Garantir que o planeador principal usa o Gemini (gerar_plano) em vez de gerar_plano_local
def planear_gemini(situacao):
    situacao = (situacao or "").strip()
    if len(situacao) < 5:
        return "Descreva a sua situação em poucas palavras. Por exemplo: *vou abrir um café*.", None
    try:
        # Usamos gerar_plano que consome o Gemini e o contexto de procura enriquecido
        plano, fontes = gerar_plano(situacao)
    except Exception as ex:
        return (f"Não consegui preparar o plano com o Gemini agora ({type(ex).__name__}: {str(ex)[:150]}). "
                "Verifique a chave de API ou tente novamente."), None
    if not plano.get("passos"):
        return NAO_SEI, None
    pdf = plano_para_pdf(plano, fontes, situacao, f"plano_{int(time.time())}.pdf")
    return plano_para_markdown(plano, fontes, situacao), pdf

# --- Estado Dinâmico do Formulário Espelho ---
# Simula os campos recolhidos pelo fluxo interativo
state_formulario = {
    "Nome Completo": "Não preenchido",
    "NIF / Documento": "Não preenchido",
    "Contacto Telefónico": "Não preenchido",
    "Morada / Localização": "Não preenchido",
    "Descrição do Pedido": "Não preenchido",
    "Ficheiros Anexados": []
}

def atualizar_espelho(nome, nif, tel, morada, desc, files):
    # Atualiza o dicionário de estado com base nos inputs recolhidos do utilizador
    state_formulario["Nome Completo"] = nome if nome else "Não preenchido"
    state_formulario["NIF / Documento"] = nif if nif else "Não preenchido"
    state_formulario["Contacto Telefónico"] = tel if tel else "Não preenchido"
    state_formulario["Morada / Localização"] = morada if morada else "Não preenchido"
    state_formulario["Descrição do Pedido"] = desc if desc else "Não preenchido"

    anexos_html = ""
    if files:
        state_formulario["Ficheiros Anexados"] = [f.name.split("/")[-1] for f in files]
        anexos_html = "".join(f"<li style='color: green;'>🟢 {name} (Pronto)</li>" for name in state_formulario["Ficheiros Anexados"])
    else:
        state_formulario["Ficheiros Anexados"] = []
        anexos_html = "<li style='color: gray;'>⚠️ Nenhum ficheiro anexado (Arraste PDFs ou Imagens acima)</li>"

    painel_html = f"""
    <div style="border: 2px solid #1a73e8; border-radius: 8px; padding: 15px; background-color: #f8f9fa;">
        <h3 style="color: #1a73e8; margin-top: 0;">🖥️ Espelho do Formulário Oficial (Em Tempo Real)</h3>
        <p style="font-size: 11px; color: #5f6368;">Os serviços municipais receberão exatamente os dados abaixo validados.</p>
        <hr style="border-top: 1px solid #dadce0;">
        <table style="width:100%; text-align:left; border-collapse: collapse;">
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0; width: 35%;">Requerente:</th><td style="padding: 8px 0;">{state_formulario['Nome Completo']}</td></tr>
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0;">NIF:</th><td style="padding: 8px 0;">{state_formulario['NIF / Documento']}</td></tr>
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0;">Telefone:</th><td style="padding: 8px 0;">{state_formulario['Contacto Telefónico']}</td></tr>
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0;">Local da Ocorrência:</th><td style="padding: 8px 0;">{state_formulario['Morada / Localização']}</td></tr>
            <tr style="border-bottom: 1px solid #f1f3f4;"><th style="padding: 8px 0;">Detalhes adicionais:</th><td style="padding: 8px 0;">{state_formulario['Descrição do Pedido']}</td></tr>
        </table>
        <h4 style="margin-bottom: 5px;">📂 Documentos e Evidências (Drag & Drop):</h4>
        <ul style="margin-top: 5px; padding-left: 20px;">
            {anexos_html}
        </ul>
    </div>
    """
    return painel_html

def submeter_ao_municipio():
    # Simula o envio direto dos dados e dos ficheiros arrastados para a API da Câmara
    if state_formulario["Nome Completo"] == "Não preenchido" or state_formulario["NIF / Documento"] == "Não preenchido":
        return gr.update(value="❌ **Erro ao Submeter**: Por favor preencha pelo menos o Nome e o NIF no painel antes de enviar.", visible=True)

    time.sleep(1.5) # Simulação de chamada de rede
    ref_processo = f"EXP-2026-{int(time.time()) % 100000:05d}"

    sucesso_msg = f"""
    ### 🎉 Candidatura Submetida com Sucesso!
    * **Número do Processo:** `{ref_processo}`
    * **Estado Atual:** `Aguardando Validação Técnica`
    * **Mensagem:** Os serviços municipais de Guimarães foram notificados e o seu espelho de formulário foi anexado com {len(state_formulario['Ficheiros Anexados'])} ficheiros.
    """
    return gr.update(value=sucesso_msg, visible=True)

# --- Interface de Utilizador ---
with gr.Blocks(title="Falar com a Câmara sem burocracia") as demo:
    gr.Markdown("# 🏛️ Balcão Aberto - Município de Guimarães\n"
                "Assistente inteligente integrado com o **Gemini**. Obtenha planos rigorosos baseados na documentação real e preencha formulários com acompanhamento interativo.")

    with gr.Tab("Perguntar ao Assistente"):
        gr.ChatInterface(
            fn=chat_fn,
            examples=[
                "Como peço o cartão municipal sénior?",
                "Quanto custa a segunda via do cartão sénior?",
                "Preciso de autorização para lançar fogo?",
            ],
        )

    with gr.Tab("Planear Cenário Complexo"):
        gr.Markdown("### 📋 Planeamento de Atividades e Obras\n"
                    "Descreva a atividade que deseja iniciar no concelho de Guimarães e o Gemini construirá uma checklist de documentos oficiais e prazos reais.")
        situacao_in = gr.Textbox(label="A minha situação", lines=2,
                                 placeholder="Ex.: vou abrir um café no centro de Guimarães")
        botao = gr.Button("Criar o meu plano", variant="primary")
        gr.Examples(
            examples=[
                "Vou abrir um café no centro de Guimarães",
                "Vou organizar uma festa com fogo de artifício",
                "Tenho 67 anos e quero apoios da Câmara",
            ],
            inputs=situacao_in,
        )
        plano_out = gr.Markdown()
        pdf_out = gr.File(label="Checklist em PDF")
        # Agora usa planear_gemini para respostas perfeitas e contextualizadas
        botao.click(planear_gemini, inputs=situacao_in, outputs=[plano_out, pdf_out])

    with gr.Tab("Preencher Formulário (Espelho)"):
        gr.Markdown("### 🖥️ Assistente de Pré-Preenchimento e Submissão Sem Burocracia\n"
                    "Insira os dados da sua candidatura ou ocorrência no formulário à esquerda. Veja o painel de validação atualizar-se à direita em tempo real.")

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("#### 📝 1. Preenchimento Interativo")
                in_nome = gr.Textbox(label="Nome Completo do Requerente", placeholder="Ex: João Silva")
                in_nif = gr.Textbox(label="NIF / Número de Identificação", placeholder="Ex: 123456789")
                in_tel = gr.Textbox(label="Contacto de Telefone", placeholder="Ex: 912345678")
                in_morada = gr.Textbox(label="Morada / Localização do Pedido", placeholder="Ex: Largo Cónego José Maria Gomes")
                in_desc = gr.TextArea(label="Informação Adicional / Descrição da Ocorrência", placeholder="Descreva em detalhe o que necessita ou o problema detetado...")

                gr.Markdown("#### 📂 2. Upload de Documentos (Drag & Drop)")
                in_files = gr.File(label="Arraste e solte fotos de evidências ou PDFs exigidos (Ex: Plantas de Localização, Documentos de Identificação)", file_count="multiple")

            with gr.Column(scale=1):
                gr.Markdown("#### 🔍 3. Painel de Validação e Envio")
                # Painel espelho dinâmico em HTML
                espelho_html = gr.HTML(value=atualizar_espelho("", "", "", "", "", None))

                # Ouvir alterações em todos os campos para atualizar o espelho dinamicamente
                campos_input = [in_nome, in_nif, in_tel, in_morada, in_desc, in_files]
                for campo in campos_input:
                    campo.change(atualizar_espelho, inputs=campos_input, outputs=espelho_html)

                botao_submeter = gr.Button("🚀 Enviar Candidatura Pronta aos Serviços Municipais", variant="primary")
                out_submissao = gr.Markdown(visible=False)
                botao_submeter.click(submeter_ao_municipio, inputs=[], outputs=out_submissao)

demo.launch(share=True)